In [ ]:
# Movie Recommendation System - Vectorization (Analysis Support)
#
# This notebook converts preprocessed movie tags into TF-IDF vectors
# for offline comparison and benchmarking workflows.
#
# Note:
# - Production serving is LSH-only in this project.
# - TF-IDF artifacts from this notebook are mainly used by Notebook 04 analysis.
#
# Input:
# - ../data/processed_movies.csv
#
# Outputs:
# - ../models/artifacts/processed_movies.pkl
# - ../models/artifacts/tfidf_vectorizer.pkl
# - ../models/artifacts/tfidf_matrix.pkl

### 1) Import required libraries

In [13]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

### 2) Load processed dataset

In [14]:
processed_data_path = Path("../data/processed_movies.csv")
movies_df = pd.read_csv("../data/processed_movies.csv")

In [15]:
print("Shape:", movies_df.shape)

Shape: (4800, 3)


In [16]:
print(movies_df.head(3))

   movie_id                                     title  \
0     19995                                    Avatar   
1       285  Pirates of the Caribbean: At World's End   
2    206647                                   Spectre   

                                                                                                                      tags  
0  in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn bet...  
1  captain barbossa, long believed to be dead, has come back to life and is headed to the edge of the earth with will t...  
2  a cryptic message from bond’s past sends him on a trail to uncover a sinister organization. while m battles politica...  


In [17]:
required_columns = {"movie_id", "title", "tags"}

In [18]:
required_columns

{'movie_id', 'tags', 'title'}

### check missing columns

In [19]:
missing_columns = required_columns.difference(movies_df.columns)

In [20]:
missing_columns

set()

# 3) Basic quality checks and cleanup before vectorization

In [21]:
rows_before = len(movies_df)
movies_df = movies_df.dropna(subset=["movie_id", "title", "tags"]).copy()
movies_df = movies_df.drop_duplicates(subset=["movie_id"]).reset_index(drop=True)
movies_df["tags"] = movies_df["tags"].astype(str).str.strip().str.lower()
movies_df = movies_df[movies_df["tags"].str.len() > 0].reset_index(drop=True)

In [22]:
rows_after = len(movies_df)

In [23]:
print("Rows before cleanup:", rows_before)
print("Rows after cleanup:", rows_after)
print("Rows removed:", rows_before - rows_after)
print("Unique movie_id count:", movies_df["movie_id"].nunique())
print("Any null tags:", movies_df["tags"].isnull().any())

Rows before cleanup: 4800
Rows after cleanup: 4800
Rows removed: 0
Unique movie_id count: 4800
Any null tags: False


### 4) Build TF-IDF vectors from tags

In [24]:
TFIDF_MAX_FEATURES = 5000

tfidf_vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    stop_words="english"
)

tfidf_matrix = tfidf_vectorizer.fit_transform(movies_df["tags"])

rows, cols = tfidf_matrix.shape
density = tfidf_matrix.nnz / (rows * cols)

In [25]:
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Non-zero entries:", tfidf_matrix.nnz)
print("Matrix density:", round(density, 6))

TF-IDF matrix shape: (4800, 5000)
Non-zero entries: 133675
Matrix density: 0.00557


### 5) Inspect vocabulary created by TF-IDF

In [26]:
feature_names = np.array(tfidf_vectorizer.get_feature_names_out())

In [27]:
print("Vocabulary size:", len(feature_names))
print("\nFirst 30 terms:")
print(feature_names[:30])

Vocabulary size: 5000

First 30 terms:
['000' '007' '10' '100' '11' '12' '13' '14' '15' '16' '17' '17th' '18'
 '18th' '19' '1930s' '1940s' '1950s' '1960s' '1970s' '1980' '1980s' '1985'
 '1990s' '19th' '19thcentury' '20' '200' '2003' '2009']


## Cosine Recommendation Utility
Create a reusable helper function to test retrieval quality directly from TF-IDF vectors.

### 6) Cosine recommendation helper

In [28]:
def recommend_cosine(movie_name, movies_data, matrix, top_n=5):
    title_series = movies_data["title"].astype(str).str.lower().str.strip()
    target = movie_name.lower().strip()

    matches = movies_data.index[title_series == target]
    if len(matches) == 0:
        contains_matches = movies_data.index[title_series.str.contains(target, regex=False)]
        if len(contains_matches) == 0:
            raise ValueError(f"Movie '{movie_name}' not found.")
        movie_index = int(contains_matches[0])
    else:
        movie_index = int(matches[0])

    cosine_scores = linear_kernel(matrix[movie_index:movie_index + 1], matrix).flatten()
    cosine_scores[movie_index] = -1.0

    top_indices = np.argsort(cosine_scores)[::-1][:top_n]
    recommendations = movies_data.iloc[top_indices][["movie_id", "title"]].copy()
    recommendations["score"] = cosine_scores[top_indices]

    return recommendations.reset_index(drop=True)

### 7) Quick recommendation test

In [29]:
sample_movie = "Avatar"
sample_recommendations = recommend_cosine(sample_movie, movies_df, tfidf_matrix, top_n=5)

print(f"Top 5 recommendations for '{sample_movie}':")
print(sample_recommendations)

Top 5 recommendations for 'Avatar':
   movie_id                    title     score
0    270938            Falcon Rising  0.203717
1     44943      Battle: Los Angeles  0.194273
2     50357                Apollo 18  0.184064
3     54138  Star Trek Into Darkness  0.171254
4     34851                Predators  0.166340


## Save Vectorization Artifacts
Persist vectors and related objects for model training and Flask inference.

### 8) Save artifacts

In [30]:
artifacts_dir = Path("../models/artifacts")
artifacts_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
processed_movies_pkl = artifacts_dir / "processed_movies.pkl"
tfidf_vectorizer_pkl = artifacts_dir / "tfidf_vectorizer.pkl"
tfidf_matrix_pkl = artifacts_dir / "tfidf_matrix.pkl"

In [31]:
with open(processed_movies_pkl, "wb") as file:
    pickle.dump(movies_df, file)

with open(tfidf_vectorizer_pkl, "wb") as file:
    pickle.dump(tfidf_vectorizer, file)

with open(tfidf_matrix_pkl, "wb") as file:
    pickle.dump(tfidf_matrix, file)


In [32]:
print("Saved artifacts:")
print("-", processed_movies_pkl)
print("-", tfidf_vectorizer_pkl)
print("-", tfidf_matrix_pkl)

Saved artifacts:
- ..\models\artifacts\processed_movies.pkl
- ..\models\artifacts\tfidf_vectorizer.pkl
- ..\models\artifacts\tfidf_matrix.pkl


### 9) Verify saved artifacts can be loaded

In [33]:
with open(processed_movies_pkl, "rb") as file:
    loaded_movies_df = pickle.load(file)

with open(tfidf_vectorizer_pkl, "rb") as file:
    loaded_vectorizer = pickle.load(file)

with open(tfidf_matrix_pkl, "rb") as file:
    loaded_tfidf_matrix = pickle.load(file)

In [34]:

print("Loaded dataframe shape:", loaded_movies_df.shape)
print("Loaded vectorizer vocabulary size:", len(loaded_vectorizer.get_feature_names_out()))
print("Loaded TF-IDF matrix shape:", loaded_tfidf_matrix.shape)

Loaded dataframe shape: (4800, 3)
Loaded vectorizer vocabulary size: 5000
Loaded TF-IDF matrix shape: (4800, 5000)
